In [1]:
# [CELL 1] STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil
import statistics

LAST_NAME = "Malitao"
STUDENT_ID = "TUPM-26-1135"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

BASE_DIR = Path.cwd()
DATASET_FILE = BASE_DIR / "user_behavior_dataset.csv"

OUTPUT_DIR = BASE_DIR

print("STUDENT CONFIGURATION")
print("---------------------")
print("Last Name:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)
print("Dataset Found:", DATASET_FILE.exists())

STUDENT CONFIGURATION
---------------------
Last Name: Malitao
Student ID: TUPM-26-1135
Seed Digit: 5
Operating System: iOS
Target Behavior Class: 1
Dataset Found: True


In [2]:
# [CELL 2] DATASET INSPECTION AND VALIDATION

required_columns = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
    "User Behavior Class"
]

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:

    reader = csv.DictReader(file)

    headers = reader.fieldnames

    print("DATASET HEADERS:", headers)

    if not headers:
        raise ValueError("The CSV file is empty or has no header.")

    missing_columns = [
        column for column in required_columns
        if column not in headers
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    total_dataset_records = sum(1 for row in reader)

print("\nDATASET VALIDATION")
print("------------------")
print("Required Columns:", required_columns)
print("Total Dataset Records:", total_dataset_records)
print("Missing Columns:", missing_columns)
print("Dataset Structure: VALID")

DATASET HEADERS: ['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

DATASET VALIDATION
------------------
Required Columns: ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)', 'User Behavior Class']
Total Dataset Records: 700
Missing Columns: []
Dataset Structure: VALID


In [3]:
# [CELL 3] FILTER MOBILE DEVICE RECORDS

selected_records = []

invalid_records = 0
matching_os_records = 0

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:

    reader = csv.DictReader(file)

    for row in reader:

        try:
            operating_system = row["Operating System"].strip()

            # Identify records matching the target operating system
            if operating_system != TARGET_OS:
                continue

            matching_os_records += 1

            behavior_class = int(row["User Behavior Class"].strip())

            # Select users belonging to the target behavior class
            if behavior_class != TARGET_BEHAVIOR_CLASS:
                continue

            battery_drain = float(row["Battery Drain (mAh/day)"].strip())
            data_usage = float(row["Data Usage (MB/day)"].strip())

            # Validate numerical values
            if battery_drain < 0 or data_usage < 0:
                invalid_records += 1
                continue

            selected_records.append({
                "User ID": row["User ID"].strip(),
                "Device Model": row["Device Model"].strip(),
                "Operating System": operating_system,
                "Battery Drain (mAh/day)": battery_drain,
                "Data Usage (MB/day)": data_usage
            })

        except (ValueError, KeyError, AttributeError, TypeError):
            invalid_records += 1


print("MOBILE DEVICE FILTERING RESULTS")
print("--------------------------------")
print("Total Dataset Records:", total_dataset_records)
print("Target Operating System:", TARGET_OS)
print("Matching Operating System:", matching_os_records)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)
print("Records in Target Behavior Class:", len(selected_records))
print("Invalid Records:", invalid_records)

MOBILE DEVICE FILTERING RESULTS
--------------------------------
Total Dataset Records: 700
Target Operating System: iOS
Matching Operating System: 146
Target Behavior Class: 1
Records in Target Behavior Class: 22
Invalid Records: 0


In [4]:
# [CELL 4] CALCULATE AVERAGE BATTERY DRAIN AND DATA USAGE

if selected_records:

    battery_values = [
        record["Battery Drain (mAh/day)"]
        for record in selected_records
    ]

    data_values = [
        record["Data Usage (MB/day)"]
        for record in selected_records
    ]

    average_battery_drain = statistics.mean(battery_values)
    average_data_usage = statistics.mean(data_values)

else:

    average_battery_drain = None
    average_data_usage = None


print("MOBILE DEVICE USAGE STATISTICS")
print("--------------------------------")
print("Selected Users:", len(selected_records))

if selected_records:
    print(f"Average Battery Drain: {average_battery_drain:.2f} mAh/day")
    print(f"Average Data Usage: {average_data_usage:.2f} MB/day")
else:
    print("No matching records found.")
    print("Average Battery Drain: N/A")
    print("Average Data Usage: N/A")

MOBILE DEVICE USAGE STATISTICS
--------------------------------
Selected Users: 22
Average Battery Drain: 467.23 mAh/day
Average Data Usage: 201.45 MB/day


In [5]:
# [CELL 5] EXPORT FILTERED CSV AND JSON SUMMARY

FILTERED_CSV = OUTPUT_DIR / "filtered_mobile_users.csv"
SUMMARY_JSON = OUTPUT_DIR / "mobile_summary.json"
BACKUP_CSV = OUTPUT_DIR / "mobile_backup.csv"

# Export selected users to CSV
fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(selected_records)


# Prepare JSON summary
summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": TARGET_OS,
    "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_dataset_records": total_dataset_records,
    "matching_operating_system_records": matching_os_records,
    "records_in_target_behavior_class": len(selected_records),
    "invalid_records": invalid_records,
    "average_battery_drain_mAh_per_day": average_battery_drain,
    "average_data_usage_MB_per_day": average_data_usage
}

# Export JSON summary
with open(SUMMARY_JSON, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)


print("FILE EXPORT RESULTS")
print("-------------------")
print("Filtered CSV:", FILTERED_CSV.name)
print("JSON Summary:", SUMMARY_JSON.name)
print("Selected Users Exported:", len(selected_records))
print("JSON Summary Created: SUCCESS")

FILE EXPORT RESULTS
-------------------
Filtered CSV: filtered_mobile_users.csv
JSON Summary: mobile_summary.json
Selected Users Exported: 22
JSON Summary Created: SUCCESS


In [6]:
# [CELL 6] BACKUP AND FILE VERIFICATION

# Create backup of filtered CSV
shutil.copy2(FILTERED_CSV, BACKUP_CSV)

# Verify exported CSV row count
with open(FILTERED_CSV, "r", newline="", encoding="utf-8") as file:

    reader = csv.DictReader(file)
    exported_count = sum(1 for row in reader)

# Verify backup row count
with open(BACKUP_CSV, "r", newline="", encoding="utf-8") as file:

    backup_reader = csv.DictReader(file)
    backup_count = sum(1 for row in backup_reader)

print("FILE VERIFICATION")
print("-----------------")

for file_path in [FILTERED_CSV, SUMMARY_JSON, BACKUP_CSV]:

    print(
        file_path.name,
        "- VERIFIED" if file_path.exists() else "- NOT FOUND"
    )

print("\nRECORD COUNT VERIFICATION")
print("-------------------------")
print("Selected Users:", len(selected_records))
print("Exported Users:", exported_count)
print("Count Matches:", exported_count == len(selected_records))

print("\nBACKUP VERIFICATION")
print("-------------------")
print("Backup Exists:", BACKUP_CSV.exists())
print("Backup Row Count:", backup_count)
print("Backup Matches:", backup_count == exported_count)
print("Backup File Size:", BACKUP_CSV.stat().st_size, "bytes")

FILE VERIFICATION
-----------------
filtered_mobile_users.csv - VERIFIED
mobile_summary.json - VERIFIED
mobile_backup.csv - VERIFIED

RECORD COUNT VERIFICATION
-------------------------
Selected Users: 22
Exported Users: 22
Count Matches: True

BACKUP VERIFICATION
-------------------
Backup Exists: True
Backup Row Count: 22
Backup Matches: True
Backup File Size: 762 bytes
